# 🔍 환경 사전 점검 (Preflight Check)

이 노트북은 RHOAI 3.5 τ-Knowledge 모델 학습을 시작하기 전에
필요한 환경 요소를 점검합니다.

## 점검 항목

| # | 항목 | 설명 |
|---|------|------|
| 1 | Python & GPU | Python 버전, CUDA 가용성, VRAM 확인 |
| 2 | 환경 변수 | `.env` 파일 로딩 및 필수 변수 존재 확인 |
| 3 | MLflow | MLflow 추적 서버 연결 상태 |
| 4 | 기본 모델 | `Qwen/Qwen3-4B-Instruct-2507` 토크나이저 로드 테스트 |

모든 항목이 ✅이면 학습을 진행할 수 있습니다.  
❌ 항목이 있으면 해당 설정을 먼저 수정하세요.

In [ ]:
"""환경 부트스트랩 — local과 workbench 모두 지원."""

import subprocess, sys
from pathlib import Path

# 프로젝트 루트 탐색 (노트북 위치 기준)
_nb_dir = Path.cwd()
_project_root = _nb_dir
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / "pyproject.toml").exists():
        _project_root = _p
        break

# 패키지 설치 확인 및 자동 설치
try:
    import rhoai_model_training_lab  # noqa: F401
    print("✅ rhoai_model_training_lab 패키지 확인됨")
except ImportError:
    print("📦 패키지 설치 중... (최초 1회)")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-e", str(_project_root)],
        stdout=subprocess.DEVNULL,
    )
    print("✅ 설치 완료 — 커널 재시작이 필요할 수 있습니다.")


In [ ]:
"""Python 버전 및 GPU 가용성 확인."""

import sys
import os

results = {}

print(f"Python 버전: {sys.version}")
py_ok = sys.version_info >= (3, 11)
results["python"] = py_ok
print(f"  Python >= 3.11: {'✅' if py_ok else '❌'}")

import platform
print(f"  플랫폼: {platform.platform()}")

# GPU / CUDA check
print()
try:
    import torch

    cuda_available = torch.cuda.is_available()
    results["cuda"] = cuda_available
    print(f"PyTorch 버전: {torch.__version__}")
    print(f"  CUDA 사용 가능: {'✅' if cuda_available else '❌'}")

    if cuda_available:
        gpu_count = torch.cuda.device_count()
        print(f"  GPU 수: {gpu_count}")
        for i in range(gpu_count):
            name = torch.cuda.get_device_name(i)
            vram_total = torch.cuda.get_device_properties(i).total_mem / (1024**3)
            vram_free = (torch.cuda.get_device_properties(i).total_mem - torch.cuda.memory_allocated(i)) / (1024**3)
            print(f"  GPU {i}: {name}")
            print(f"    총 VRAM: {vram_total:.1f} GB")
            print(f"    가용 VRAM: {vram_free:.1f} GB")
            results["gpu_name"] = name
    else:
        print("  ℹ️  GPU 없음 — 데이터 준비/검증은 CPU만으로 수행 가능합니다.")
except ImportError:
    results["cuda"] = None
    print("ℹ️  PyTorch 미설치 — 데이터 준비 단계에서는 필요하지 않습니다.")
    print("   LoRA/OSFT 학습 시 GPU workbench에서 자동 설치됩니다.")


In [ ]:
"""Check .env loaded and required environment variables present."""

import os
from pathlib import Path

from rhoai_model_training_lab.config import load_env, PROJECT_ROOT

# Load .env
env_path = PROJECT_ROOT / ".env"
env_exists = env_path.exists()
results["env_file"] = env_exists
print(f".env 파일 존재: {'✅' if env_exists else '❌'} ({env_path})")

if env_exists:
    load_env(env_path)
else:
    print("  ⚠️  .env.example 을 .env로 복사하고 값을 설정하세요.")
    print(f"  cp {PROJECT_ROOT / '.env.example'} {env_path}")

# Required variables for learner training path
required_vars = {
    "BASE_MODEL_ID": "기본 모델 ID",
    "TOKENIZER_ID": "토크나이저 ID",
}

# Optional but recommended
optional_vars = {
    "MLFLOW_TRACKING_URI": "MLflow 추적 URI",
    "S3_ENDPOINT": "S3 엔드포인트",
    "S3_BUCKET": "S3 버킷",
    "BASE_SERVING_ENDPOINT": "기본 모델 서빙 엔드포인트",
}

print("\n--- 필수 환경 변수 ---")
all_required_ok = True
for var, desc in required_vars.items():
    val = os.environ.get(var, "")
    ok = bool(val)
    if not ok:
        all_required_ok = False
    status = "✅" if ok else "❌"
    display_val = val[:40] + "..." if len(val) > 40 else val
    print(f"  {status} {var} ({desc}): {display_val or '(미설정)'}")

results["env_vars"] = all_required_ok

print("\n--- 선택 환경 변수 ---")
for var, desc in optional_vars.items():
    val = os.environ.get(var, "")
    status = "✅" if val else "⚠️"
    display_val = val[:40] + "..." if len(val) > 40 else val
    print(f"  {status} {var} ({desc}): {display_val or '(미설정)'}")

In [2]:
"""Check working directories exist and are writable."""

work_dirs = {
    "data": PROJECT_ROOT / "data",
    "models": PROJECT_ROOT / "models",
    "checkpoints": PROJECT_ROOT / "checkpoints",
}

print("--- 작업 디렉토리 점검 ---")
all_dirs_ok = True
for name, p in work_dirs.items():
    exists = p.exists()
    writable = os.access(p, os.W_OK) if exists else False

    if exists and writable:
        status = "✅"
    elif exists:
        status = "⚠️  (읽기 전용)"
        all_dirs_ok = False
    else:
        try:
            p.mkdir(parents=True, exist_ok=True)
            status = "✅ (새로 생성)"
        except OSError as exc:
            status = f"❌ (생성 실패: {exc})"
            all_dirs_ok = False

    print(f"  {status} {name}: {p}")

results["work_dirs"] = all_dirs_ok


NameError: name 'PROJECT_ROOT' is not defined

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/hyochoi/dev/rhoai-model-training-lab/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 566, in _log_error
    f.result()
  File "/Users/hyochoi/dev/rhoai-model-training-lab/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 584, in shell_channel_thread_main
    _, msg2 = self.session.feed_identities(msg, copy=False)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/hyochoi/dev/rhoai-model-training-lab/.venv/lib/python3.12/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/hyochoi/dev/rhoai-model-training-lab/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 566, in _log_error
    f.result()
  File "/Users/hyo

In [ ]:
"""MLflow 추적 서버 연결 점검."""

mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
mlflow_ok = False

print("--- MLflow 연결 점검 ---")

if not mlflow_uri:
    print("  ⚠️  MLFLOW_TRACKING_URI가 설정되지 않았습니다.")
    print("  .env 파일을 확인하세요.")
else:
    print(f"  MLflow URI: {mlflow_uri}")
    try:
        import httpx

        resp = httpx.get(
            f"{mlflow_uri.rstrip('/')}/api/2.0/mlflow/experiments/search",
            params={"max_results": "1"},
            timeout=10,
            verify=False,
        )
        if resp.status_code == 200:
            mlflow_ok = True
            data = resp.json()
            exp_count = len(data.get("experiments", []))
            print(f"  ✅ MLflow 연결 성공 (실험 수: {exp_count})")
        elif resp.status_code in (401, 403):
            mlflow_ok = True  # 서버는 도달 가능
            print(f"  ✅ MLflow 서버 도달 가능 (인증 필요: HTTP {resp.status_code})")
            print("  ℹ️  Workbench에서 실행 시 ServiceAccount 토큰으로 자동 인증됩니다.")
        else:
            print(f"  ⚠️  MLflow HTTP {resp.status_code}")
    except Exception as exc:
        print(f"  ❌ MLflow 연결 실패: {exc}")

results["mlflow"] = mlflow_ok


In [ ]:
"""Check base model accessibility — tokenizer load test."""

model_id = os.environ.get("BASE_MODEL_ID", "Qwen/Qwen3-4B-Instruct-2507")
model_ok = False

print("--- 기본 모델 접근성 점검 ---")
print(f"  모델 ID: {model_id}")

try:
    from transformers import AutoTokenizer

    print("  토크나이저 로딩 중...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model_ok = True

    print(f"  ✅ 토크나이저 로드 성공")
    print(f"    어휘 크기: {tokenizer.vocab_size:,}")
    print(f"    모델 최대 길이: {getattr(tokenizer, 'model_max_length', 'N/A')}")

    # Chat template check
    has_chat_template = hasattr(tokenizer, "chat_template") and tokenizer.chat_template
    print(f"    채팅 템플릿: {'✅ 있음' if has_chat_template else '⚠️  없음'}")

    # Quick encode/decode test
    test_text = "Hello, I'd like to check my account balance."
    tokens = tokenizer.encode(test_text)
    decoded = tokenizer.decode(tokens)
    print(f"    인코딩/디코딩 테스트: ✅ ({len(tokens)} tokens)")

    # Tool call template test
    test_messages = [
        {"role": "system", "content": "You are a banking assistant."},
        {"role": "user", "content": "What is my balance?"},
    ]
    try:
        formatted = tokenizer.apply_chat_template(test_messages, tokenize=False)
        print(f"    채팅 템플릿 적용: ✅ ({len(formatted)} chars)")
    except Exception as tmpl_err:
        print(f"    채팅 템플릿 적용: ⚠️  {tmpl_err}")

except ImportError:
    print("  ❌ transformers 패키지가 설치되지 않았습니다.")
except Exception as exc:
    print(f"  ❌ 토크나이저 로드 실패: {exc}")
    print("  모델에 대한 네트워크 접근이 필요하거나 로컬 캐시를 확인하세요.")

results["model"] = model_ok

In [ ]:
"""최종 결과 요약 테이블."""

from rich.console import Console
from rich.table import Table

console = Console()

table = Table(title="🔍 환경 사전 점검 결과", show_header=True)
table.add_column("항목", style="bold")
table.add_column("상태")
table.add_column("비고")

checks = [
    ("Python >= 3.11", results.get("python", False), f"v{sys.version_info.major}.{sys.version_info.minor}"),
    ("CUDA / GPU", results.get("cuda", None), results.get("gpu_name", "데이터 준비 시 불필요")),
    (".env 파일", results.get("env_file", False), ""),
    ("필수 환경 변수", results.get("env_vars", False), ""),
    ("작업 디렉토리", results.get("work_dirs", False), ""),
    ("MLflow", results.get("mlflow", False), mlflow_uri or "미설정"),
    ("기본 모델", results.get("model", False), model_id),
]

all_pass = True
critical_fail = False
for name, ok, note in checks:
    if ok is None:
        status = "⚠️ 선택"
        style = "yellow"
    elif ok:
        status = "✅ 통과"
        style = "green"
    else:
        status = "❌ 실패"
        style = "red"
    table.add_row(name, f"[{style}]{status}[/{style}]", str(note))
    if ok is False:
        all_pass = False
        if name in ("Python >= 3.11", "필수 환경 변수"):
            critical_fail = True

console.print(table)
print()

if all_pass:
    print("🎉 모든 사전 점검 항목을 통과했습니다!")
    print("   데이터 준비: data_preparation/01_prepare_tau_sources.ipynb")
    print("   학습 (GPU workbench): 03_lora_finetuning.ipynb")
elif critical_fail:
    print("🚫 필수 항목에서 실패가 있습니다. 위의 안내를 따라 수정하세요.")
else:
    print("⚠️  일부 항목을 확인하세요.")
    print("   GPU가 없으면 데이터 준비/검증까지 진행 가능합니다.")
    print("   LoRA/OSFT 학습은 GPU workbench에서 수행하세요.")
